# ECIS — Ollama on Colab

Three extractors, same pipeline: **Llama 3.1 8B**, **Mistral 7B**, **Qwen2.5 14B** (4-bit). CLI: `--model llama`, `--model mistral`, `--model qwen`, `--model both` (Llama+Mistral), `--model all` (all three).

**Mode A** — run the pipeline on Colab. **Mode B** — tunnel Ollama to the machine that runs the CLI.

Runtime → Change runtime type → GPU (A100 preferred for Qwen 14B).

## Setup

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed.")

In [ ]:
import subprocess, time

proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f"Ollama server started (PID: {proc.pid})")

In [ ]:
!ollama pull llama3.1:8b-instruct-q8_0
!ollama pull mistral:7b-instruct
!ollama pull qwen2.5:14b-instruct-q4_K_M
print("Llama 3.1 8B, Mistral 7B, and Qwen2.5 14B are ready.")

In [ ]:
import requests

resp = requests.get("http://localhost:11434/api/tags", timeout=10)
models = [m["name"] for m in resp.json().get("models", [])]
print(f"Available models: {models}")

have_llama = any("llama3.1" in m for m in models)
have_mistral = any("mistral" in m for m in models)
have_qwen = any("qwen" in m for m in models)
print(f"Llama: {have_llama}  Mistral: {have_mistral}  Qwen: {have_qwen}")
if not (have_llama and have_mistral and have_qwen):
    raise RuntimeError(
        "Need llama3.1:8b-instruct-q8_0, mistral:7b-instruct, and qwen2.5:14b-instruct-q4_K_M"
    )

## Mode A: pipeline on Colab

Mount Drive (repo at `MyDrive/Mycroft_Contribution`) **or** unzip `Mycroft_Contribution.zip` into `/content/`.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

candidates = [
    Path("/content/Mycroft_Contribution"),
    Path("/content/drive/MyDrive/Mycroft_Contribution"),
]
REPO_ROOT = next((p for p in candidates if (p / "src" / "ecis").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repo not found. Unzip Mycroft_Contribution.zip under /content/ or put it on Drive."
    )

PROJECT_PATH = REPO_ROOT / "src" / "ecis"
WORK_DIR = str(REPO_ROOT)
os.chdir(WORK_DIR)
os.environ["PYTHONPATH"] = str(REPO_ROOT / "src")
print(f"Working directory: {WORK_DIR}")

In [ ]:
import os
from pathlib import Path

os.chdir(WORK_DIR)
os.environ["PYTHONPATH"] = str(Path(WORK_DIR) / "src")

!pip install -q -r requirements.txt
!pip install -e .
!python -m spacy download en_core_web_sm 2>/dev/null
print("Dependencies installed.")

In [ ]:
import re
from pathlib import Path

env_path = Path(WORK_DIR) / "src" / "ecis" / ".env"
example = Path(WORK_DIR) / "src" / "ecis" / ".env.example"
if not env_path.exists():
    env_path.write_text(example.read_text() if example.exists() else "")

content = env_path.read_text()
replacements = {
    r"OLLAMA_BASE_URL=.*": 'OLLAMA_BASE_URL="http://localhost:11434"',
    r"LLM_MODEL=.*": 'LLM_MODEL="llama3.1:8b-instruct-q8_0"',
    r"LLM_LLAMA_MODEL=.*": 'LLM_LLAMA_MODEL="llama3.1:8b-instruct-q8_0"',
    r"LLM_MISTRAL_MODEL=.*": 'LLM_MISTRAL_MODEL="mistral:7b-instruct"',
    r"LLM_QWEN_MODEL=.*": 'LLM_QWEN_MODEL="qwen2.5:14b-instruct-q4_K_M"',
}
for pattern, value in replacements.items():
    if re.search(pattern, content):
        content = re.sub(pattern, value, content)
    else:
        content = content.rstrip() + "\n" + value + "\n"
env_path.write_text(content)

In [ ]:
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --init-db

In [ ]:
TICKER = "TICKER"
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKER} --model llama
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKER} --model mistral
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKER} --model qwen

In [ ]:
EXTRACT_MODEL = "all"  # llama | mistral | qwen | both | all
TICKERS = "TICKER"
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKERS} --model {EXTRACT_MODEL}

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path(WORK_DIR) / "src" / "ecis" / "data" / "db" / "signals.db"
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
total = conn.execute("SELECT COUNT(*) as n FROM signals").fetchone()["n"]
print(f"Total signals: {total}")
cols = {row[1] for row in conn.execute("PRAGMA table_info(signals)").fetchall()}
if "llm_model" in cols:
    for r in conn.execute(
        "SELECT COALESCE(llm_model, 'unknown') as model, COUNT(*) as n FROM signals GROUP BY model ORDER BY n DESC"
    ):
        print(f"  {r['model']}: {r['n']}")
conn.close()

## Mode B: Cloudflare tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("Cloudflared installed.")

In [ ]:
import subprocess, time, re

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(8)
output = ""
match = None
while True:
    chunk = tunnel_proc.stderr.read1(8192).decode()
    if not chunk:
        break
    output += chunk
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", output)
    if match:
        break

if match:
    url = match.group()
    print(f'OLLAMA_BASE_URL="{url}"')
    print()
    print("Put that URL in src/ecis/.env, and keep these tags:")
    print('LLM_LLAMA_MODEL="llama3.1:8b-instruct-q8_0"')
    print('LLM_MISTRAL_MODEL="mistral:7b-instruct"')
    print('LLM_QWEN_MODEL="qwen2.5:14b-instruct-q4_K_M"')
    print()
    print("From the repo root on the CLI machine:")
    print("python -m ecis.main --extract --ticker TICKER --model llama")
    print("python -m ecis.main --extract --ticker TICKER --model mistral")
    print("python -m ecis.main --extract --ticker TICKER --model qwen")
    print("python -m ecis.main --extract --ticker TICKER --model all")
    print()
    print("Keep this notebook running while extraction is in progress.")
else:
    print("Could not find tunnel URL.")
    print(output)

In [ ]:
import requests

try:
    resp = requests.get(f"{url}/api/tags", timeout=15)
    models = [m["name"] for m in resp.json().get("models", [])]
    print(f"Tunnel is live. Models: {models}")
except Exception as e:
    print(f"Tunnel check failed: {e}")

In [ ]:
import time, requests

while True:
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print(".", end="", flush=True)
    except Exception:
        print("x", end="", flush=True)
    time.sleep(60)